# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/T0othIess/FlyRank-AI-ML-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*  
I chose logistic regression as the first model because it is interpretable and produces ranking scores. More complex models could be tested later, but complexity should only be added if it improves the same ranking metric under the same validation design.   

The feature set is small, with a mixture of numeric and categorical variables, so a simple interpretable model is an appropriate first choice.

In [3]:
import os
from huggingface_hub import login
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get("HF_TOKEN")
login(HF_TOKEN)
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

fact_content_daily_performance_table = con.sql(f"SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')")
dim_content_table = con.sql(f"SELECT * from read_parquet('{rel}/dim_content.parquet')")


#time for some explanation:
#first of all: we grouped by client id and content id without report date BECAUSE the old dataframe was counting for each page each day, meaning multiple rows could have the same page
#but just on different days, that created noise at the end of ranking because at the top there could be the same page repeated just on different days
#the way to solve it is to group by client and content alone, which means for each page on a certain client we get the data that happened across the ENTIRE month, not just per day

#which brings us to a 2nd explanation about why we used sum() on each column
#the reason is because we used group by, in SQL, columns have 2 options when we are using group by:
#1- be included in the group by clause (like client id and content id)
#2- get aggregated with SUM, AVG, ANY_VALUE, and what not
#if u leave it as is like doing f.gsc_avg_position as a column, it would give an error
#thats why for some columns that have the same value for all rows across the days (like d.main_intent for example), we used ANY_VALUE
#because it satisfies the condition for columns when using GROUP BY, its just used for columns that have the same value across the rows

#AS FOR THE EQUATION ON AVG_POSITION:
#one of the issues we had is that avg_position gets played a lot on low impressions, basically impressions matter a lot
#thats why we have to do a weighted avg, not a normal average, where the weight in the equation is impressions.
#if u dont understand u can look up what the formula for weighted average is
df = con.sql("""SELECT f.client_hash_id, f.content_hash_id, SUM(f.gsc_clicks) AS total_clicks, SUM(f.gsc_impressions) AS total_impressions,
                SUM(f.gsc_avg_position * f.gsc_impressions) *1.0 / SUM(f.gsc_impressions) AS weighted_avg_position,
                ANY_VALUE(d.search_volume) AS search_volume, ANY_VALUE(d.competition_level) AS competition_level, ANY_VALUE(d.main_intent) AS main_intent
                FROM fact_content_daily_performance_table AS f JOIN dim_content_table AS d USING(content_hash_id)
                WHERE
                    f.gsc_data_available IS TRUE AND d.is_deleted IS FALSE AND f.gsc_avg_position >0 AND d.search_volume IS NOT NULL AND d.competition_level IS NOT NULL
                GROUP BY f.client_hash_id, f.content_hash_id ORDER BY f.client_hash_id, f.content_hash_id""").df()

test = df["total_impressions"]
print("Impression threshold calculations: ")
print(f"Min: {test.min()}")
print(f"Median: {test.median()}")
print(f"Mean: {test.mean()}")
print(f"Max: {test.max()}\n")

#percentile explained:
#the value stored on the left is the value used in the percentile
#percentile rule example: 50th of impressions have <= X
#the numpy percentile function finds us the X and we store it in p50 for example
p50 = np.percentile(test, 50)
p70 = np.percentile(test, 70)
p90 = np.percentile(test, 90)
p95 = np.percentile(test, 95)
p99 = np.percentile(test, 99)

print(f"50th: {p50}")
print(f"70th: {p70}")
print(f"90th: {p90}")
print(f"95th: {p95}")
print(f"99th: {p99}")

thresholds = [10,50,100,150,200,300,500,1000]

for threshold in thresholds:
    #to explain the logic rq, (df["total_impressions"] >= threshold) returns a series wtih true and falses, sum calculates only the true's (since they are 1 in binary) to give the count
    print(f"Amount of pages at {threshold}-> {(df["total_impressions"] >= threshold).sum()}")

#the whole point of what was above is to help me determine a minimum impressions limit to help me remove noise that happened in w04 where the top 10 all had like 1-10 impressions
df = df.query("total_impressions >= 200").reset_index(drop=True)

df["main_intent"] = df["main_intent"].astype("category")

competition_ranks = pd.api.types.CategoricalDtype(categories=["LOW", "MEDIUM", "HIGH"], ordered=True)
df["competition_level"] = df["competition_level"].astype(competition_ranks)
df["position_tier"] = pd.cut(df["weighted_avg_position"], bins=[0,10,20,float("inf")], labels=["page_1", "striking", "page_3_5"])
expected_ctr_per_tier = df.groupby("position_tier")["total_clicks"].sum() / df.groupby("position_tier")["total_impressions"].sum()
df["expected_ctr"] = df["position_tier"].map(expected_ctr_per_tier).astype(float)
df["ctr"] = df["total_clicks"] / df["total_impressions"]
df["ctr_gap"] = df["expected_ctr"] - df["ctr"]

#to determine the threshold that will define is_high_gap label
test = df["ctr_gap"]
print("\nctr_gap threshold calculations: ")
print(f"Min: {test.min()}")
print(f"Median: {test.median()}")
print(f"Mean: {test.mean()}")
print(f"Max: {test.max()}\n")

p50 = np.percentile(test, 50)
p70 = np.percentile(test, 70)
p90 = np.percentile(test, 90)
p95 = np.percentile(test, 95)
p99 = np.percentile(test, 99)

print(f"50th: {p50}")
print(f"70th: {p70}")
print(f"90th: {p90}")
print(f"95th: {p95}")
print(f"99th: {p99}")

truth_threshold = 0.002
df["is_high_gap"] = (df["ctr_gap"] >= truth_threshold).astype(int)

position_map = {"page_1": 0, "striking": 1, "page_3_5": 2}
competition_map = {"LOW": 0, "MEDIUM": 1, "HIGH":2}
df["position_tier_enc"] = df["position_tier"].map(position_map)
df["competition_level_enc"] = df["competition_level"].map(competition_map)

c:\Users\M-H-M-D\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Impression threshold calculations: 
Min: 1.0
Median: 227.0
Mean: 1746.8509751405184
Max: 617124.0

50th: 227.0
70th: 870.0
90th: 4385.0
95th: 7878.0
99th: 23049.99000000002
Amount of pages at 10-> 135873
Amount of pages at 50-> 111409
Amount of pages at 100-> 97940
Amount of pages at 150-> 89200
Amount of pages at 200-> 82507
Amount of pages at 300-> 73065
Amount of pages at 500-> 60934
Amount of pages at 1000-> 44602

ctr_gap threshold calculations: 
Min: -0.07593411487998898
Median: 0.0013089443369939046
Mean: 0.0002479355301214112
Max: 0.0033762299475972256

50th: 0.0013089443369939046
70th: 0.002105397364010823
90th: 0.0033762299475972256
95th: 0.0033762299475972256
99th: 0.0033762299475972256


Feature/threshold choices, worked out from the data itself (not guessed):

Impression threshold (≥200): chosen by checking percentiles of `total_impressions` and how many pages survive at each candidate threshold — this fixes the w04 problem where the top of the queue was dominated by 1-2 impression noise.  
`ctr_gap` threshold (0.002) for `is_high_gap`: chosen by checking percentiles of `ctr_gap` itself — 0.002 sits close to the 70th percentile, so roughly the top 30% of pages by gap size count as "high gap."  

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
# GroupShuffleSplit works like train_test_split, but takes a "groups" argument
# that keeps all rows belonging to the same group (here, same client) on the
# same side of the split - either all in train, or all in test, never both.
# n_splits=1 means we only want ONE version of this split (not multiple, which
# GroupShuffleSplit also supports for things like cross-validation).
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_hash_id"]
gss = GroupShuffleSplit(test_size = 0.2, n_splits=1, random_state=1)

feature_list = ["weighted_avg_position", "position_tier_enc", "search_volume", "competition_level_enc"]
X = df[feature_list]
y = df["is_high_gap"]

# Step 2 (gss = GroupShuffleSplit(...)) only set up the RULES (80/20 ratio, seed, how many splits). It didn't touch our actual data at all.
# .split(X, y, groups=groups) is the step that actually DOES the splitting.
# this is where our real data finally gets divided according to those rules.
# The only quirk: because GroupShuffleSplit can generate multiple different splits (for cross-validation), .split() hands back a generator instead of
# a plain answer, it produces splits one at a time, on request.
# Since we only want 1 split, next() grabs that single result.
train_idx, test_idx = next(gss.split(X,y, groups=groups))

print(f"Total rows: {len(X)}")
print(f"Train rows: {len(train_idx)}({len(train_idx)/len(X):.2%}), Test rows: {len(test_idx)}({round(len(test_idx)/len(X)* 100, 2)}%)")

#the results of the split were: Train rows: 81263(98.49%), Test rows: 1244(1.51%)
#as we have noticed, test rows only has 1.51% instead of the presumed 20%, thats because since we grouped by clients, some clients have a lot of pages under them, thats why its uneven
#the way to solve it is to find the seed that gives the correct split

best_seed = None
best_diff = float("inf")
best_fraction = None
for seed in range(1, 51):
    gss_try = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_idx_try, test_idx_try = next(gss_try.split(X, y, groups=groups))
    fraction_try = len(test_idx_try) / len(X)
    diff = abs(fraction_try - 0.2)
    if diff < best_diff:
        best_diff = diff
        best_seed = seed
        best_fraction = fraction_try

print(f"Best seed: {best_seed}, test fraction: {best_fraction:%}")
gss = GroupShuffleSplit(test_size =0.2, n_splits = 1, random_state = best_seed)
train_idx, test_idx = next(gss.split(X,y, groups= groups))
print(f"Train rows: {len(train_idx)}({len(train_idx)/len(X):.2%}), Test rows: {len(test_idx)}({len(test_idx)/len(X):.2%})")

#the reason we didnt use df.loc is cuz loc is for index labels, iloc is for the integer position.
train_clients = set(df.iloc[train_idx]["client_hash_id"])
test_clients = set(df.iloc[test_idx]["client_hash_id"])

print(f"Clients shared by train and test: {len(train_clients & test_clients)}")

Total rows: 82507
Train rows: 81263(98.49%), Test rows: 1244(1.51%)
Best seed: 49, test fraction: 19.952246%
Train rows: 66045(80.05%), Test rows: 16462(19.95%)
Clients shared by train and test: 0


Because clients have unequal numbers of pages, `GroupShuffleSplit(test_size=0.2)` does not guarantee that the test set will contain exactly 20% of all rows. With the initial seed, the selected test clients contained only 1.51% of page rows. I searched a fixed seed range from 1 to 50 only to find a grouped split whose row fraction was close to 20%; seed 49 produced a 19.95% test-row fraction.  

The final split contains 66,045 training rows (80.05%) and 16,462 test rows (19.95%). The client grouping rule remained intact: no client appears in both training and test data.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*  
The model and baseline are evaluated on the same grouped-client test set. The logistic regression model is trained only on the training clients, then predicts the probability that each test page has a high CTR gap.  

The baseline ranks the same test pages by observed `ctr_gap`, from largest to smallest. Because `ctr_gap` is the quantity used to define `is_high_gap`, this is an oracle baseline: it has direct access to the answer and should reach 100% precision@k by construction. It is included as a transparent upper-bound reference, not as a realistic deployable method.  

The learned model does not use `ctr_gap`, clicks, impressions, CTR, or expected CTR as features. It ranks pages using only position, position tier, search volume, and competition level.  

In [57]:
from sklearn.linear_model import LogisticRegression
from IPython.display import display #this to make it possible to print styler objects
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

#initialziing the model
#logistic regression learns by iteration, and setting the max to 1000 is common
model = LogisticRegression(max_iter=1000, random_state=1)

#training the model
model.fit(X_train, y_train)

#model.predict_proba(X_test) returns an array of shape (n_test_samples, 2) where column 0 is probability of class 0 (is_high_gap = 0)
#and column 1 is probability of class 1 (is_high_gap = 1)
#to explain [:,1], in numpy : means get all rows, 1 means column 1 of the rows
#so basically it gets me a 1d array of the column 1 values for all the rows
y_pred_proba = model.predict_proba(X_test)[:,1]

test_df = df.loc[X_test.index].copy()
test_df["is_high_gap"] = y_test.values
test_df["predicted_high_gap"] = y_pred_proba

test_df_model = test_df.sort_values("predicted_high_gap", ascending=False)
test_df_baseline = test_df.sort_values("ctr_gap", ascending=False)

#to find what the random selection base rate is
#since y_test consists of the is_high_gap column which is just 1s and 0s, mean finds the base rate
base_random = y_test.mean() 

data = {"k": [], "Random Prediction Precision@k": [], "Baseline Precision@k": [], "Model A Precision@k": []}
for k in [20,50,100,200,500,1000]:
    data["k"].append(k)
    data["Random Prediction Precision@k"].append(base_random)

    top_k_model = test_df_model.head(k)
    model_precision_at_k = top_k_model["is_high_gap"].sum() / k
    data["Model A Precision@k"].append(model_precision_at_k)

    top_k_baseline = test_df_baseline.head(k)
    baseline_precision_at_k = top_k_baseline["is_high_gap"].sum() / k
    data["Baseline Precision@k"].append(baseline_precision_at_k)

#note: results isnt a dataframe, its a styler object, because we used .style
results = pd.DataFrame(data).style.hide(axis="index").format("{:.1%}", subset=["Random Prediction Precision@k", "Baseline Precision@k", "Model A Precision@k"])\
.set_properties(**{"text-align": "center"}) 
display(results)

k,Random Prediction Precision@k,Baseline Precision@k,Model A Precision@k
20,42.6%,100.0%,55.0%
50,42.6%,100.0%,64.0%
100,42.6%,100.0%,57.0%
200,42.6%,100.0%,50.5%
500,42.6%,100.0%,45.8%
1000,42.6%,100.0%,41.0%


### Results interpretation

The grouped-client test set has a high-gap base rate of 42.6%, meaning that a randomly selected test page would be expected to have a 42.6% chance of meeting the `is_high_gap` definition.  

Model A improves on random selection near the top of the review queue. It reaches 55.0% precision@20 and 64.0% precision@50, meaning that 11 of the top 20 pages and 32 of the top 50 pages are true high-gap pages. The model remains above the random expected rate through k=500, but its precision falls to 41.0% at k=1000, slightly below the 42.6% base rate.  

The oracle baseline reaches 100% precision at every k because it ranks pages using `ctr_gap`, the exact quantity used to define `is_high_gap`. It is therefore an upper-bound reference rather than a realistic deployable method. Model A does not see `ctr_gap`, CTR, clicks, or impressions; it uses only pre-click search context to prioritize pages for review.  

The initial model (Model A) treats `position_tier` and `competition_level` as ordinal integers, e.g. page_1=0, striking=1, page_3_5=2. This implies equal spacing between categories: the model assumes the difference between page_1 and striking is the same as between striking and page_3_5, which is not necessarily true.

To address this, Model B replaces the ordinal encoding of `position_tier` with one-hot encoding. One-hot encoding creates a separate binary column for each category, allowing the model to learn an independent coefficient for each position tier instead of assuming a fixed linear relationship between them. It also log-transforms `search_volume` (which is heavily right-skewed).  

In [58]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

#creates the one-hot encoder object
#the reason for sparse_output=False is so that the position_encoded has type numpy.ndarry which works with DataFrame, cuz without it it would be a sprase smthn type,
#which doesnt work and raises an error.
ohe = OneHotEncoder(sparse_output=False)

categories = ["position_tier"]
#fit it on the training data to learn the categories and transforms it (meaning it starts filling up the matrix with 1s and 0s)
position_encoded = ohe.fit_transform(df[categories])

#get the column names, reason why it was on ohe not position_encoded because position_encoded is a matrix without column names, ohe creates the column names
position_columns = ohe.get_feature_names_out(categories)

#the reason for index= df.index is to ensure the dataframe doesnt mix up cuz it will be added to df, so it must have it's index
position_df = pd.DataFrame(data=position_encoded, columns=position_columns, index=df.index)
df[position_columns] = position_df

#basically because search_volume is heavily skewed, we use log for it so that the model doesnt depend too much on it basically
#log1p(x) means log(1+x), reason for the 1+ is cuz x can be 0, to not get a math error
df["log_search_volume"] = np.log1p(df["search_volume"])
features_list_B = ["weighted_avg_position", "log_search_volume", "competition_level_enc"] + position_columns.tolist()

#df.loc[rows, columns] thats the syntax
X_B_train = df.loc[X_train.index, features_list_B]
X_B_test = df.loc[X_test.index, features_list_B]
y_B_train = y_train
y_B_test = y_test

model_B = LogisticRegression(max_iter=1000, random_state=1)
model_B.fit(X_B_train, y_B_train)
y_B_pred_proba = model_B.predict_proba(X_B_test)[:,1]

test_df_B = df.loc[X_B_test.index].copy()
test_df_B["is_high_gap"] = y_B_test.values
test_df_B["predicted_high_gap"] = y_B_pred_proba

test_df_model_B = test_df_B.sort_values("predicted_high_gap", ascending=False)

data = {"k": [], "Model A precision@k": [], "Model B precision@k": []}
for k in [20,50,100,200,500,1000]:
    data["k"].append(k)
    top_k_model_A = test_df_model.head(k)
    model_A_precision_at_k = top_k_model_A["is_high_gap"].sum() / k
    data["Model A precision@k"].append(model_A_precision_at_k)


    top_k_model_B = test_df_model_B.head(k)
    model_B_precision_at_k = top_k_model_B["is_high_gap"].sum() / k
    data["Model B precision@k"].append(model_B_precision_at_k)

#note: results isnt a dataframe, its a styler object, because we used .style
results = pd.DataFrame(data).style.hide(axis="index").format("{:.1%}", subset=["Model B precision@k", "Model A precision@k"]).set_properties(**{"text-align": "center"}) 
display(results)


k,Model A precision@k,Model B precision@k
20,55.0%,95.0%
50,64.0%,78.0%
100,57.0%,72.0%
200,50.5%,67.5%
500,45.8%,62.6%
1000,41.0%,60.5%


### Model B results

Model B improves on Model A at every k. It reaches 95.0% precision@20 and 72.0% precision@100, meaning 19 of the top 20 pages and 72 of the top 100 pages are true high-gap pages. Even at k=1000 it holds 60.5%, well above the 42.6% base rate.  

The one-hot encoding of `position_tier` and the log transform of `search_volume` drive the improvement. One-hot encoding lets the model learn an independent coefficient for each position tier instead of forcing a fixed linear spacing, and the log transform reduces the influence of a few extremely high search-volume pages.  

The oracle baseline still reaches 100% precision at every k, since it ranks by `ctr_gap` itself. Model B remains below this upper bound because it never sees `ctr_gap`, CTR, clicks, or impressions — it uses only pre-click search context to prioritize pages for review.  

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*  

The model is a ranking tool, so errors are defined by queue position rather than a probability cutoff. Using a review queue of k=1000, the top 1000 ranked pages contain 395 false positives (ranked into the queue but not high-gap) and miss 6,403 high-gap pages that fall below the queue.  

The large number of missed pages is mostly a capacity effect: there are roughly 7,000 high-gap pages in the test set but only 1,000 queue slots, so most high-gap pages necessarily fall below the cutoff. The more telling result is that the model's ranking only weakly tracks true severity: the Spearman correlation between the model's score and `ctr_gap` is 0.201. The model reliably fills the queue with high-gap pages (60.5% precision vs a 42.6% base rate) but does not strongly sort them by how badly they need review.  

The 395 false positives are concentrated in the `striking` tier (369 of 395), with 26 in `page_1` and none in `page_3_5`. Most (351 of 395) have `LOW` competition. Because the model leans heavily on position tier, it flags pages in the tiers it associates with high gap even when they are not actually high-gap.  

The model leans almost entirely on position tier. The `page_1` and `striking` coefficients are strongly positive (+1.98 and +1.96), while `page_3_5` is strongly negative (−6.91). Search volume and competition level have only small effects. This explains the false-positive pattern: because the model leans on `striking`, striking pages near the boundary get flagged as high-gap.  

In [ ]:
from scipy.stats import spearmanr

#model ranking pages as high gap but they arent.
false_positive = test_df_model_B.head(1000).query("is_high_gap == 0")

#model ranking pages as not high gap but they are.
false_negative = test_df_model_B.iloc[999:].query("is_high_gap == 1")

print(f"amount of rows of the false positives: {len(false_positive)}")
print(f"amount of rows of the false negatives: {len(false_negative)}")

print(f"\n{false_positive["position_tier"].value_counts()}")
print(f"\n{false_positive["competition_level"].value_counts()}")

columns = ["content_hash_id", "total_clicks", "total_impressions", "weighted_avg_position","position_tier",  "search_volume", "competition_level", "main_intent", "expected_ctr",
           "ctr_gap", "is_high_gap", "predicted_high_gap"]

# The high-gap pages the model DID catch in the top 1000 (true positives).
true_positive = test_df_model_B.head(1000).query("is_high_gap == 1")

# Compare ctr_gap between the caught and missed high-gap pages.
comparison = pd.DataFrame({
    "group": ["caught (true positive)", "missed (false negative)"],
    "count": [len(true_positive), len(false_negative)],
    "mean_ctr_gap": [true_positive["ctr_gap"].mean(), false_negative["ctr_gap"].mean()],
    "median_ctr_gap": [true_positive["ctr_gap"].median(), false_negative["ctr_gap"].median()],
    "max_ctr_gap": [true_positive["ctr_gap"].max(), false_negative["ctr_gap"].max()],
})
display(comparison)

# How well does the model's ranking track true ctr_gap?
#to explain this spearmanr shi, its simple
#it compared preicted_high_gap with ctr_gap to answer one question "do these two things tend to move up together?"
#dont worry about p, ignore it, we just need corr
corr, p = spearmanr(test_df_model_B["predicted_high_gap"], test_df_model_B["ctr_gap"])
print(f"Spearman correlation: {corr:.3f}")

# After training, sklearn stores each feature's learned coefficient in model_B.coef_. It's a 2D array with one row (since this is a single binary model),
# so [0] grabs that row of one coefficient per feature.
# A positive coefficient means: as the feature increases, the model is more likely to flag the page as high-gap. Negative means the opposite.
# Larger magnitude = stronger influence.

# We wrap the coefficients in a Series labeled with the feature names so
# the output is readable, then sort from most negative to most positive.
#why did we use series? because a series is best used for single column tables, which is what we want here
feature_coefficients = pd.Series(model_B.coef_[0],index=features_list_B).sort_values()
display(feature_coefficients.to_frame("coefficient"))


amount of rows of the false positives: 395
amount of rows of the false negatives: 6403



position_tier
striking    369
page_1       26
page_3_5      0
Name: count, dtype: int64

competition_level
LOW       351
MEDIUM     25
HIGH       19
Name: count, dtype: int64


,group,count,mean_ctr_gap,median_ctr_gap,max_ctr_gap
0,caught (true positive),605,0.003068,0.003182,0.003376
1,missed (false negative),6403,0.002964,0.003182,0.003376


Spearman correlation: 0.201


,coefficient
position_tier_page_3_5,-6.911404
competition_level_enc,-0.096054
weighted_avg_position,0.048802
log_search_volume,0.060836
position_tier_striking,1.956145
position_tier_page_1,1.979352


## Self-check

Before you submit, confirm each line honestly:

- [ x ] Every section above is filled — markdown thinking AND the code that backs it
- [ x ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x ] No client names, URLs, or private queries anywhere
- [ x ] My claims use careful words: observed, measured, directional, decision-support
- [ x ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.